In [26]:
import pandas as pd
import numpy as np
import nltk
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json
import random
from bert_score import score

In [ ]:
# load json dataset
file_path = "../../data/Hallucination/qa_data.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = [json.loads(line) for line in file]

In [33]:
random.seed(42)

records = []
hallucination_flags = []
answers = []

# Sample 1000 random entries from data
sampled_entries = random.sample(data, 1000)

for entry in sampled_entries:
    is_hallucinated = random.choice([True, False])
    actual_output = entry["hallucinated_answer"] if is_hallucinated else entry["right_answer"]    
    records.append({
        "input": entry["question"],
        "actual_output": actual_output,
        "context": entry["knowledge"]
    })
    hallucination_flags.append(1 if is_hallucinated else 0)
    answers.append(actual_output)

# Convert to DataFrame
df = pd.DataFrame(records).reset_index()

hallucination_df = pd.DataFrame({
    "index": df["index"],
    "hallucinated_flag": hallucination_flags,
    "actual_output": answers,
    "context": df["context"]
})

In [34]:
qa_data_eval = hallucination_df.to_dict(orient='records')

In [35]:
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')

In [36]:
# Function to calculate METEOR score
def calculate_meteor(reference, hypothesis):
    reference_tokens = nltk.word_tokenize(reference)
    hypothesis_tokens = nltk.word_tokenize(hypothesis)
    return meteor_score([reference_tokens], hypothesis_tokens)

# Function to calculate ROUGE score
def calculate_rouge(reference, hypothesis, n=1):
    scorer = rouge_scorer.RougeScorer([f'rouge{n}'], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)
    return scores[f'rouge{n}'].fmeasure  # Return F1-score

# Function to calculate BLEU score
def calculate_bleu(reference, hypothesis, n=2):
    weights = [1.0 / n] * n  # Distribute weight equally among n-grams
    return sentence_bleu([reference.split()], hypothesis.split(), weights=weights)

In [37]:
config = {
    'ROUGE':[1,2,'L'],
    'BLEU':[i for i in range(1,6)],
    'METEOR':'D'
}

In [38]:
results = {
    key: {
        value: {
            'scores': [],
            'thresholds':{
                round(i * 0.1, 1): {
                    'preds':[],
                    'accuracy': 0,
                    'precision':0,
                    'recall':0,
                    'f1':0
                    } for i in range(11)
                },
        } 
        for value in values
    }
    for key, values in config.items()
}

In [39]:
y_true = []

for entry in qa_data_eval:
    
    reference = entry['context']
    hallucinated = entry['actual_output']
    true_label = entry['hallucinated_flag']
    
    y_true.append(true_label)
    
    for method in results:
        
        for param in results[method]:
            
            if method == 'ROUGE':
                score = calculate_rouge(reference, hallucinated, n=param)
                
            if method == 'BLEU':
                score = calculate_bleu(reference, hallucinated, n=param)
                
            if method == 'METEOR':
                score = calculate_meteor(reference, hallucinated)
                
            results[method][param]['scores'].append(score)

In [40]:
for method in results:
    
    for param in results[method]:
        
        for score in results[method][param]['scores']:
            
            for threshold in results[method][param]['thresholds']:
                
                results[method][param]['thresholds'][threshold]['preds'].append(1 if score <= threshold else 0)
                
        for threshold in results[method][param]['thresholds']:
            
            results[method][param]['thresholds'][threshold]['accuracy'] = \
                accuracy_score(y_true, results[method][param]['thresholds'][threshold]['preds'])
                
            results[method][param]['thresholds'][threshold]['precision'] = \
                precision_score(y_true, results[method][param]['thresholds'][threshold]['preds'], zero_division=0)
                
            results[method][param]['thresholds'][threshold]['recall'] = \
                recall_score(y_true, results[method][param]['thresholds'][threshold]['preds'], zero_division=0)

            results[method][param]['thresholds'][threshold]['f1'] = \
                f1_score(y_true, results[method][param]['thresholds'][threshold]['preds'], zero_division=0)

In [41]:
data = []

# Traverse the results dictionary to extract the required values
for method, method_data in results.items():
    for param, param_data in method_data.items():
        for threshold, threshold_data in param_data['thresholds'].items():
            
            accuracy = round(threshold_data['accuracy'], 3)
            precision = round(threshold_data['precision'], 3)
            recall = round(threshold_data['recall'], 3)
            f1 = round(threshold_data['f1'], 3)
            
            data.append({
                'method': method,
                'param': param,
                'threshold': threshold,
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1
            })

# Create a DataFrame
df = pd.DataFrame(data)

In [44]:
df.loc[df.groupby("method")["f1"].idxmax()]


,method,param,threshold,accuracy,precision,recall,f1
38,BLEU,1,0.5,0.472,0.472,1.0,0.641
94,METEOR,D,0.6,0.472,0.472,1.0,0.641
7,ROUGE,1,0.7,0.472,0.472,1.0,0.641


## BERTScores

In [15]:
# load json dataset
file_path = "../../data/Hallucination/qa_data.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = [json.loads(line) for line in file]
    
random.seed(42)

records = []
hallucination_flags = []
answers = []

# Sample 1000 random entries from data
sampled_entries = random.sample(data, 1000)

for entry in sampled_entries:
    is_hallucinated = random.choice([True, False])
    actual_output = entry["hallucinated_answer"] if is_hallucinated else entry["right_answer"]    
    records.append({
        "input": entry["question"],
        "actual_output": actual_output,
        "context": entry["knowledge"]
    })
    hallucination_flags.append(1 if is_hallucinated else 0)
    answers.append(actual_output)

# Convert to DataFrame
df = pd.DataFrame(records).reset_index()

hallucination_df = pd.DataFrame({
    "index": df["index"],
    "hallucinated": hallucination_flags,
    "actual_output": answers
})

In [10]:
import time

# Prepare inputs
candidates = df["actual_output"].tolist()
references = df["context"].tolist()
bert_models = ["bert-base-uncased", "roberta-large"]

# Run BERTScore sequentially and time each model
for model_name in bert_models:
    print(f"\nRunning BERTScore for: {model_name}...")
    start_time = time.time()

    P, R, F1 = score(
        candidates,
        references,
        lang="en",
        model_type=model_name,
        rescale_with_baseline=True  # Optional but improves comparability
    )

    end_time = time.time()
    elapsed = end_time - start_time
    print(f"{model_name} completed in {elapsed:.2f} seconds")

    # Save to DataFrame
    df[f"{model_name}_bertscore_precision"] = P.tolist()
    df[f"{model_name}_bertscore_recall"] = R.tolist()
    df[f"{model_name}_bertscore_f1"] = F1.tolist()


Running BERTScore for: bert-base-uncased...
bert-base-uncased completed in 103.77 seconds

Running BERTScore for: roberta-large...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


roberta-large completed in 664.42 seconds


In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

thresholds = np.arange(0.0, 1.01, 0.01)

# Store results for each model
all_metrics = {}

for model in bert_models:
    f1_col = f"{model}_bertscore_f1"
    metrics_list = []

    for threshold in thresholds:
        # Predict hallucination based on threshold
        df["predicted_hallucination"] = (df[f1_col] < threshold).astype(int)

        y_true = hallucination_df["hallucinated"]
        y_pred = df["predicted_hallucination"]

        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        metrics_list.append({
            "threshold": round(threshold, 3),
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1
        })

    metrics_df = pd.DataFrame(metrics_list)
    all_metrics[model] = metrics_df

    best_f1_row = metrics_df.loc[metrics_df["accuracy"].idxmax()]
    print(f"\nBest threshold for {model} by accuracy:")
    print(best_f1_row)


Best threshold for bert-base-uncased by accuracy:
threshold    0.910000
accuracy     0.466000
precision    0.466000
recall       1.000000
f1_score     0.635744
Name: 91, dtype: float64

Best threshold for roberta-large by accuracy:
threshold    0.860000
accuracy     0.466000
precision    0.466000
recall       1.000000
f1_score     0.635744
Name: 86, dtype: float64


In [13]:
all_metrics

{'bert-base-uncased':      threshold  accuracy  precision    recall  f1_score
 0         0.00     0.287   0.095082  0.062232  0.075227
 1         0.01     0.268   0.096970  0.068670  0.080402
 2         0.02     0.254   0.106742  0.081545  0.092457
 3         0.03     0.243   0.109920  0.087983  0.097735
 4         0.04     0.235   0.111688  0.092275  0.101058
 ..         ...       ...        ...       ...       ...
 96        0.96     0.466   0.466000  1.000000  0.635744
 97        0.97     0.466   0.466000  1.000000  0.635744
 98        0.98     0.466   0.466000  1.000000  0.635744
 99        0.99     0.466   0.466000  1.000000  0.635744
 100       1.00     0.466   0.466000  1.000000  0.635744
 
 [101 rows x 5 columns],
 'roberta-large':      threshold  accuracy  precision    recall  f1_score
 0         0.00     0.204   0.194444  0.225322  0.208748
 1         0.01     0.201   0.204263  0.246781  0.223518
 2         0.02     0.205   0.212914  0.261803  0.234841
 3         0.03     0.2